# Motor Insurance Claim Prediction — Step 1: Load & Inspect

**Project:** Machine Learning Based Prediction of Motor Insurance Claim
**Group 2, Team 5**



In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

train = pd.read_csv('/content/train.csv')
test = pd.read_csv('data/test.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'data/test.csv'

## 1.1 First look at the data

In [ ]:
train.head()

In [ ]:
test.head()

## 1.2 Column consistency between train and test

In [ ]:
train_cols = set(train.columns)
test_cols = set(test.columns)

print("In train but not test:", train_cols - test_cols)
print("In test but not train:", test_cols - train_cols)


## 1.3 Data types

In [ ]:
dtype_summary = pd.DataFrame({
    'dtype': train.dtypes,
    'n_unique': train.nunique(),
    'sample_value': train.iloc[0]
})
dtype_summary


## 1.4 Missing values

Proposal claims zero missing values across all fields — verifying that here for both train and test.

In [ ]:
missing_train = train.isnull().sum()
missing_test = test.isnull().sum()

print("Missing values in train:")
print(missing_train[missing_train > 0] if missing_train.sum() > 0 else "None found — 0 missing values across all columns.")
print()
print("Missing values in test:")
print(missing_test[missing_test > 0] if missing_test.sum() > 0 else "None found — 0 missing values across all columns.")


## 1.5 Identifier check

`policy_id` should be unique per row in both train and test, and the two sets should not overlap in IDs (no leakage).

In [ ]:
print("Train policy_id unique:", train['policy_id'].is_unique, "| count:", train['policy_id'].nunique())
print("Test policy_id unique: ", test['policy_id'].is_unique, "| count:", test['policy_id'].nunique())
print("Overlap between train and test IDs:", len(set(train['policy_id']) & set(test['policy_id'])))


## 1.6 Target variable distribution (`is_claim`)

Confirming the class imbalance flagged in the proposal (~93.6% / 6.4%).

In [ ]:
target_counts = train['is_claim'].value_counts().sort_index()
target_pct = train['is_claim'].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({'count': target_counts, 'percent': target_pct.round(2)})
target_summary.index = ['No claim (0)', 'Claim (1)']
target_summary


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5,4))
target_counts.plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452'])
ax.set_xticklabels(['No claim (0)', 'Claim (1)'], rotation=0)
ax.set_ylabel('Number of policies')
ax.set_title('Target variable distribution — is_claim')
for i, v in enumerate(target_counts):
    ax.text(i, v + 500, f"{v:,}", ha='center')
plt.tight_layout()
plt.show()


## 1.7 Split predictor columns into numeric vs categorical

Quick inventory before we get into cleaning/feature engineering in Step 2.

In [ ]:
non_predictive = ['policy_id']
target_col = 'is_claim'

predictor_cols = [c for c in train.columns if c not in non_predictive + [target_col]]

numeric_cols = train[predictor_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = train[predictor_cols].select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Numeric predictors ({len(numeric_cols)}):")
print(numeric_cols)
print()
print(f"Categorical / object predictors ({len(categorical_cols)}):")
print(categorical_cols)


**Note:** a few columns are numeric-looking but stored as text (`max_torque`, `max_power` — e.g. `"91Nm@4250rpm"`), so they currently land in the categorical bucket above. These will be parsed into numeric fields in Step 2, along with the Yes/No flag columns which are also currently `object` dtype.

In [ ]:
# Preview unique value counts for categorical columns to spot high-cardinality fields early
cat_cardinality = train[categorical_cols].nunique().sort_values(ascending=False)
cat_cardinality


## 1.8 Quick numeric summary

In [ ]:
train[numeric_cols].describe().T


In [ ]:
print("=== Step 1 Summary ===")
print(f"Train rows: {train.shape[0]:,} | Test rows: {test.shape[0]:,}")
print(f"Predictor columns: {len(predictor_cols)} (+ policy_id, + is_claim in train only)")
print(f"Numeric predictors: {len(numeric_cols)} | Categorical/object predictors: {len(categorical_cols)}")
print(f"Missing values -> train: {train.isnull().sum().sum()}, test: {test.isnull().sum().sum()}")
print(f"policy_id unique in train: {train['policy_id'].is_unique}, in test: {test['policy_id'].is_unique}")
print(f"Train/Test policy_id overlap: {len(set(train['policy_id']) & set(test['policy_id']))}")
print(f"Target distribution -> No claim: {target_pct.iloc[0]:.2f}% | Claim: {target_pct.iloc[1]:.2f}%")


## 1.9 What Step 1 confirms

- No missing values in either file
- `policy_id` is unique in both sets with no overlap — safe to drop as a predictor
- Target is imbalanced at roughly 93.6% / 6.4% (no-claim / claim), confirming the proposal's numbers
- Several categorical-looking columns are really Yes/No flags or text-encoded numerics (`max_torque`, `max_power`) that need parsing
- **`test.csv` has no target column**, so it can't be used to evaluate models — Section 1.10 below sets it aside and builds our own labeled validation set from `train.csv`


## 1.10 Create our own train/validation split

`test.csv` has no `is_claim` column, so it can't be used to evaluate model performance — we're setting it aside and won't use it going forward.

Instead, we'll carve out a labeled validation set from `train.csv` itself using an **80-20 stratified split**, so we always have ground truth to check Precision/Recall/F1/ROC-AUC against. Stratifying on `is_claim` keeps the ~93.6% / 6.4% class ratio consistent in both halves.


In [ ]:
from sklearn.model_selection import train_test_split

# From here on, "train"/"test" refer to our own split of train.csv — the original test.csv is set aside.
X = train.drop(columns=['is_claim'])
y = train['is_claim']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)


### 1.11 Confirm the split preserved the class ratio

In [ ]:
split_check = pd.DataFrame({
    'train_full (train.csv)': train['is_claim'].value_counts(normalize=True).sort_index() * 100,
    'X_train (80%)': y_train.value_counts(normalize=True).sort_index() * 100,
    'X_test (20%)': y_test.value_counts(normalize=True).sort_index() * 100,
}).round(2)
split_check.index = ['No claim (0)', 'Claim (1)']
split_check
